In [ ]:
# Colab 進行matplotlib繪圖時顯示繁體中文
# 下載台北思源黑體並命名taipei_sans_tc_beta.ttf，移至指定路徑
!wget -O TaipeiSansTCBeta-Regular.ttf https://drive.google.com/uc?id=1eGAsTN1HBpJAkeVM57_C7ccp7hbgSz3_&export=download

import matplotlib

# 改style要在改font之前
# plt.style.use('seaborn')

matplotlib.font_manager.fontManager.addfont('TaipeiSansTCBeta-Regular.ttf')
matplotlib.rc('font', family='Taipei Sans TC Beta')

In [ ]:
import pandas as pd
import numpy as np
from scipy.sparse import load_npz
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, classification_report, f1_score
import pandas as pd
import numpy as np
from scipy.sparse import load_npz
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
import matplotlib.pyplot as plt

#資料（只保留漲跌）
X_sparse = load_npz("5d_3_trend_vector_model.npz")
y_df = pd.read_csv("5d_3_trend_y.csv", header=None).rename(columns={0: "trend"})

valid_idx = y_df[y_df["trend"].isin([-1, 1])].index
X_binary = X_sparse[valid_idx]
y_binary = y_df.loc[valid_idx, "trend"].replace({-1: 0, 1: 1}).values  # 轉成 0=跌, 1=漲



#切分訓練與測試
X_train, X_test, y_train, y_test = train_test_split(
    X_binary, y_binary, test_size=0.2, stratify=y_binary, random_state=42
)

#嘗試六種基本模型
models = {
    "NB": MultinomialNB(),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "SVM": SVC(kernel="linear"),
    "DT": DecisionTreeClassifier(),
    "RF": RandomForestClassifier(class_weight="balanced", random_state=42),
    "XGB": XGBClassifier(use_label_encoder=False, eval_metric='mlogloss')
}


results = {}

for name, model in models.items():
    if name == "NB":
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
    else:
        model.fit(X_train.toarray(), y_train)
        y_pred = model.predict(X_test.toarray())

    cm = confusion_matrix(y_test, y_pred, labels=[1, 0])
    acc = accuracy_score(y_test, y_pred)
    results[name] = {"conf": cm, "acc": acc}

#結果
for name, info in results.items():
    cm = info["conf"]
    acc = info["acc"]

    table_data = [
        ["", "真實為漲", "真實為跌"],
        ["預測為漲", cm[0][0], cm[0][1]],
        ["預測為跌", cm[1][0], cm[1][1]]
    ]


    fig, ax = plt.subplots(figsize=(4, 2.5))
    ax.axis("off")
    table = ax.table(cellText=table_data, loc="center", cellLoc="center", colWidths=[0.2]*3)
    table.auto_set_font_size(False)
    table.set_fontsize(12)
    table.scale(1.2, 1.2)
    plt.title(f"{name} 正確率: {acc*100:.2f}%", fontsize=14, color="red", pad=10)
    plt.show()

In [ ]:
import pandas as pd
import numpy as np
from scipy.sparse import load_npz
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.naive_bayes import MultinomialNB
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score
from tqdm import tqdm

# 資料（只保留漲跌）
X_sparse = load_npz("5d_3_trend_vector_model.npz")
y_df = pd.read_csv("5d_3_trend_y.csv", header=None).rename(columns={0: "trend"})
time_df = pd.read_csv("5d_3_trend_time.csv", header=None).rename(columns={0: "datetime"})
time_df.columns = ["datetime"]
time_df["datetime"] = pd.to_datetime(time_df["datetime"])
time_df["id"] = time_df.index

valid_idx = y_df[y_df["trend"].isin([-1, 1])].index
X_binary = X_sparse[valid_idx]
y_binary = y_df.loc[valid_idx, "trend"].replace({-1: 0, 1: 1}).values
time_binary = time_df.loc[valid_idx].reset_index(drop=True)


#時間資料處理
df_all = pd.DataFrame({
    "id": time_binary["id"].values,
    "datetime": pd.to_datetime(time_binary["datetime"].values),
    "label": y_binary
})
df_all["row_id"] = df_all.index
df_all["datetime"] = pd.Series(time_binary["datetime"].values).dt.floor("D")
df_all.set_index("datetime", inplace=True)

print(f"時間範圍：{df_all.index.min()} ~ {df_all.index.max()}")
print(f"資料筆數：{len(df_all)}")


#移動回測
confidence_threshold = 0.3
start_date = df_all.index.min() + pd.DateOffset(months=3)
end_date = df_all.index.max()
results = []


for current_day in tqdm(pd.date_range(start=start_date, end=end_date, freq="D")):
    train_start = current_day - pd.DateOffset(months=3)
    train_end = current_day - pd.DateOffset(days=1)
    test_day = current_day

    train_ids = df_all[(df_all.index >= train_start) & (df_all.index <= train_end)]["row_id"].values
    test_ids = df_all[df_all.index == test_day]["row_id"].values


    if len(train_ids) == 0 or len(test_ids) == 0:
        continue  #無資料跳過

    X_train = X_binary[train_ids].toarray()
    y_train = y_binary[train_ids]
    X_test = X_binary[test_ids].toarray()
    clf = RandomForestClassifier(class_weight="balanced", random_state=42)
    clf.fit(X_train, y_train)
    probas = clf.predict_proba(X_test)

    for i, idx in enumerate(test_ids):
        prob = probas[i]
        prob_diff = abs(prob[1] - prob[0])
        true_label = y_binary[idx]
        if prob_diff < confidence_threshold:
            results.append({"id": idx, "predict": "no_action", "true": true_label, "datetime": df_all.iloc[idx].name})
        else:
            pred_label = clf.predict(X_binary[idx].toarray())[0]
            results.append({"id": idx, "predict": pred_label, "true": true_label, "datetime": df_all.iloc[idx].name})

#結果
result_df = pd.DataFrame(results)
out_df = result_df[result_df["predict"] != "no_action"].copy()
out_df["predict"] = out_df["predict"].astype(int)
out_df["true"] = out_df["true"].astype(int)

y_true = out_df["true"]
y_pred = out_df["predict"]

print("Confusion Matrix（出手樣本）:")
print(confusion_matrix(y_true, y_pred, labels=[1, 0]))
print("\nClassification Report（出手樣本）:")
print(classification_report(y_true, y_pred, target_names=["漲 (1)", "跌 (0)"]))
print(f"\n出手次數：{len(out_df)} / 全部預測次數：{len(result_df)}")
print(f"Macro F1-score: {f1_score(y_true, y_pred, average='macro'):.4f}")
print(f"Accuracy: {accuracy_score(y_true, y_pred):.4f}")

In [ ]:
cm = confusion_matrix(y_true, y_pred, labels=[1, 0])
acc = accuracy_score(y_true, y_pred)

table_data = [
        ["", "真實為漲", "真實為跌"],
        ["預測為漲", cm[0][0], cm[0][1]],
        ["預測為跌", cm[1][0], cm[1][1]]]


fig, ax = plt.subplots(figsize=(4, 2.5))
ax.axis("off")
table = ax.table(cellText=table_data, loc="center", cellLoc="center", colWidths=[0.2]*3)
table.auto_set_font_size(False)
table.set_fontsize(12)
table.scale(1.2, 1.2)
plt.title(f"正確率: {acc*100:.2f}%", fontsize=14, color="red", pad=10)
plt.show()

In [ ]:
daily_acc = out_df.groupby("datetime").apply(lambda g: (g["true"] == g["predict"]).mean())

plt.figure(figsize=(12, 5))
plt.plot(daily_acc.index, daily_acc.values, label="每日準確率", color="royalblue")
plt.axhline(y=daily_acc.mean(), color="brown", linestyle="--", label=f"平均準確率：{daily_acc.mean():.3f}")
plt.title("每日預測準確率（僅含出手樣本）")
plt.ylabel("Accuracy")
plt.xlabel("Date")
plt.xticks(rotation=45)
plt.ylim(0, 1.05)
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
from scipy.sparse import load_npz
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
import matplotlib.pyplot as plt

#資料（只保留漲跌）
X_sparse = load_npz("5d_3_trend_vector_model.npz")
y_df = pd.read_csv("5d_3_trend_y.csv", header=None).rename(columns={0: "trend"})

valid_idx = y_df[y_df["trend"].isin([-1, 1])].index
X_binary = X_sparse[valid_idx]
y_binary = y_df.loc[valid_idx, "trend"].replace({-1: 0, 1: 1}).values

#移除過短文章
min_terms = 5
word_counts = X_binary.getnnz(axis=1)

valid_mask = word_counts >= min_terms
X_filtered = X_binary[valid_mask]
y_filtered = y_binary[valid_mask]

print(f"原始資料數：{X_binary.shape[0]}")
print(f"移除過短文章後剩下：{X_filtered.shape[0]} 篇（刪除 {X_binary.shape[0] - X_filtered.shape[0]} 篇）")


#切訓練與測試集
X_train, X_test, y_train, y_test = train_test_split(
    X_filtered, y_filtered, test_size=0.2, stratify=y_filtered, random_state=42
)

#模型訓練與預測
clf = RandomForestClassifier(class_weight="balanced", random_state=42)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test.toarray())

#結果
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred, labels=[1, 0]))

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=["跌 (0)", "漲 (1)"]))

macro_f1 = f1_score(y_test, y_pred, average='macro')
accuracy = accuracy_score(y_test, y_pred)

print(f"\n Macro F1-score: {macro_f1:.4f}")
print(f"\n正確率: {accuracy:.4f}")

In [ ]:
#資料（只保留漲跌）
X_sparse = load_npz("5d_3_trend_vector_model.npz")
y_df = pd.read_csv("5d_3_trend_y.csv", header=None).rename(columns={0: "trend"})

valid_idx = y_df[y_df["trend"].isin([-1, 1])].index
X_binary = X_sparse[valid_idx]
y_binary = y_df.loc[valid_idx, "trend"].replace({-1: 0, 1: 1}).values  # 轉成 0=跌, 1=漲

#刪除過短文章
min_terms = 5
word_counts = X_binary.getnnz(axis=1)

valid_mask = word_counts >= min_terms
X_filtered = X_binary[valid_mask]
y_filtered = y_binary[valid_mask]

print(f"原始資料數：{X_binary.shape[0]}")
print(f"移除過短文章後剩下：{X_filtered.shape[0]} 篇（刪除 {X_binary.shape[0] - X_filtered.shape[0]} 篇）")

#chi-square特徵選擇
selector = SelectKBest(score_func=chi2, k=2500)
X_selected = selector.fit_transform(X_filtered, y_filtered)

#切訓練與測試集
X_train, X_test, y_train, y_test = train_test_split(
    X_selected, y_filtered, test_size=0.2, stratify=y_filtered, random_state=42
)

#模型訓練與預測
clf = RandomForestClassifier(class_weight="balanced", random_state=42)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test.toarray())

#結果
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred, labels=[1, 0]))

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=["跌 (0)", "漲 (1)"]))

macro_f1 = f1_score(y_test, y_pred, average='macro')
accuracy = accuracy_score(y_test, y_pred)

print(f"\n Macro F1-score: {macro_f1:.4f}")
print(f"\n正確率: {accuracy:.4f}")

In [ ]:
import pandas as pd
import numpy as np
from scipy.sparse import load_npz
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import f1_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import seaborn as sns

# === 1. 載入並預處理（只留漲跌、過濾短文）===
X_sparse = load_npz("5d_3_trend_vector_model.npz")
y_df = pd.read_csv("5d_3_trend_y.csv", header=None).rename(columns={0: "trend"})

valid_idx = y_df[y_df["trend"].isin([-1, 1])].index
X_binary = X_sparse[valid_idx]
y_binary = y_df.loc[valid_idx, "trend"].replace({-1: 0, 1: 1}).values

# 移除過短文章
word_counts = X_binary.getnnz(axis=1)
valid_mask = word_counts >= 5
X_filtered = X_binary[valid_mask]
y_filtered = y_binary[valid_mask]

# 切分資料
X_train, X_test, y_train, y_test = train_test_split(
    X_filtered, y_filtered, test_size=0.2, stratify=y_filtered, random_state=42
)

#模型設定
models = {
    "Random Forest": (RandomForestClassifier(class_weight="balanced"), {
        "n_estimators": [100, 200, 300],
        "max_depth": [10, 20, None]
    }),
    "SVM": (SVC(probability=True), {
        "C": [0.1, 1, 5],
        "kernel": ["linear"]
    }),
    "Naive Bayes": (MultinomialNB(), {
        "alpha": [0.5, 1.0]
    }),
    "KNN": (KNeighborsClassifier(), {
        "n_neighbors": [3, 5, 7]
    }),
    "Decision Tree": (DecisionTreeClassifier(), {
        "max_depth": [10, 20, None]
    }),
    "XGBoost": (XGBClassifier(use_label_encoder=False, eval_metric='logloss',grow_policy='lossguide'), {
        "n_estimators": [100, 200, 300],
        "max_depth": [3, 6, 10],
    })
}

#尋找最佳參數
results = []

for name, (model, param_grid) in models.items():
    print(f"搜尋 {name} 的最佳參數")
    grid = GridSearchCV(model, param_grid, cv=3, scoring="accuracy", n_jobs=-1)

    if name == "Naive Bayes":
        grid.fit(X_train, y_train)
        y_pred = grid.best_estimator_.predict(X_test)
    else:
        grid.fit(X_train.toarray(), y_train)
        y_pred = grid.best_estimator_.predict(X_test.toarray())

    acc = accuracy_score(y_test, y_pred)
    results.append({
        "Model": name,
        "Best Params": grid.best_params_,
        "Accuracy": acc
    })

    # === 表格形式的混淆矩陣 ===
    cm = confusion_matrix(y_test, y_pred, labels=[1, 0])
    table_data = [
        ["", "真實為漲", "真實為跌"],
        ["預測為漲", cm[0][0], cm[0][1]],
        ["預測為跌", cm[1][0], cm[1][1]]
    ]

    fig, ax = plt.subplots(figsize=(4, 2.5))
    ax.axis("off")
    table = ax.table(cellText=table_data, loc="center", cellLoc="center", colWidths=[0.2]*3)
    table.auto_set_font_size(False)
    table.set_fontsize(12)
    table.scale(1.2, 1.2)
    plt.title(f"{name} - 正確率: {acc*100:.2f}%", fontsize=14, color="red", pad=10)
    plt.show()

# === 4. 顯示結果 ===
results_df = pd.DataFrame(results).sort_values(by="Accuracy", ascending=False)
print("模型排名：")
print(results_df.to_string(index=False))


In [ ]:
from sklearn.ensemble import VotingClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from scipy.sparse import load_npz
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score

#資料（只保留漲跌）
X_sparse = load_npz("5d_3_trend_vector_model.npz")
y_df = pd.read_csv("5d_3_trend_y.csv", header=None).rename(columns={0: "trend"})

valid_idx = y_df[y_df["trend"].isin([-1, 1])].index
X_binary = X_sparse[valid_idx]
y_binary = y_df.loc[valid_idx, "trend"].replace({-1: 0, 1: 1}).values

#移除過短文章
word_counts = X_binary.getnnz(axis=1)
valid_mask = word_counts >= 5
X_filtered = X_binary[valid_mask]
y_filtered = y_binary[valid_mask]

#切訓練與測驗集
X_train, X_test, y_train, y_test = train_test_split(
    X_filtered, y_filtered, test_size=0.2, stratify=y_filtered, random_state=42
)

#表現前四好的模型
rf = RandomForestClassifier(n_estimators=200, max_depth=None, class_weight="balanced", random_state=42)
nb = MultinomialNB(alpha=0.5)
svm = SVC(C=0.1, kernel="linear", probability=True)
xgb = XGBClassifier(n_estimators=200, max_depth=6, use_label_encoder=False, eval_metric='logloss')

#組合
ensemble = VotingClassifier(
    estimators=[
        ('rf', rf),
        ('nb', nb),
        ('svm', svm),
        ('xgb', xgb)
    ],
    voting='soft',
    n_jobs=-1
)

#訓練與預測
ensemble.fit(X_train.toarray(), y_train)
y_pred = ensemble.predict(X_test.toarray())



#結果
print("RF + NB + SVM + XGB")


print("\n分類報告:")
print(classification_report(y_test, y_pred, target_names=["漲 (1)", "跌 (0)"]))

macro_f1 = f1_score(y_test, y_pred, average='macro')
accuracy = accuracy_score(y_test, y_pred)

print(f"\nMacro F1-score: {macro_f1:.4f}")
print(f"Accuracy: {accuracy:.4f}")

cm = confusion_matrix(y_test, y_pred, labels=[1, 0])
acc = accuracy_score(y_test, y_pred)

table_data = [
        ["", "真實為漲", "真實為跌"],
        ["預測為漲", cm[0][0], cm[0][1]],
        ["預測為跌", cm[1][0], cm[1][1]]]


fig, ax = plt.subplots(figsize=(4, 2.5))
ax.axis("off")
table = ax.table(cellText=table_data, loc="center", cellLoc="center", colWidths=[0.2]*3)
table.auto_set_font_size(False)
table.set_fontsize(12)
table.scale(1.2, 1.2)
plt.title(f"正確率: {acc*100:.2f}%", fontsize=14, color="red", pad=10)
plt.show()

In [ ]:
#表現前三好的模型
ensemble = VotingClassifier(
    estimators=[
        ('rf', rf),
        ('nb', nb),
        #('svm', svm),
        ('xgb', xgb)
    ],
    voting='soft',
    n_jobs=-1
)

#模型訓練與預測
ensemble.fit(X_train.toarray(), y_train)
y_pred = ensemble.predict(X_test.toarray())

#結果
print("RF + NB + XGB")


print("\n分類報告:")
print(classification_report(y_test, y_pred, target_names=["漲 (1)", "跌 (0)"]))

macro_f1 = f1_score(y_test, y_pred, average='macro')
accuracy = accuracy_score(y_test, y_pred)

print(f"\n Macro F1-score: {macro_f1:.4f}")
print(f" Accuracy: {accuracy:.4f}")

cm = confusion_matrix(y_test, y_pred, labels=[1, 0])
acc = accuracy_score(y_test, y_pred)

table_data = [
        ["", "真實為漲", "真實為跌"],
        ["預測為漲", cm[0][0], cm[0][1]],
        ["預測為跌", cm[1][0], cm[1][1]]]


fig, ax = plt.subplots(figsize=(4, 2.5))
ax.axis("off")
table = ax.table(cellText=table_data, loc="center", cellLoc="center", colWidths=[0.2]*3)
table.auto_set_font_size(False)
table.set_fontsize(12)
table.scale(1.2, 1.2)
plt.title(f"正確率: {acc*100:.2f}%", fontsize=14, color="red", pad=10)
plt.show()


In [ ]:
import pandas as pd
import numpy as np
from scipy.sparse import load_npz
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.naive_bayes import MultinomialNB
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score
from tqdm import tqdm

#資料（只保留漲和跌）
X_sparse = load_npz("5d_3_trend_vector_model.npz")
y_df = pd.read_csv("5d_3_trend_y.csv", header=None).rename(columns={0: "trend"})
time_df = pd.read_csv("5d_3_trend_time.csv", header=None).rename(columns={0: "datetime"})
time_df.columns = ["datetime"]
time_df["datetime"] = pd.to_datetime(time_df["datetime"])
time_df["id"] = time_df.index


valid_idx = y_df[y_df["trend"].isin([-1, 1])].index
X_binary = X_sparse[valid_idx]
y_binary = y_df.loc[valid_idx, "trend"].replace({-1: 0, 1: 1}).values
time_binary = time_df.loc[valid_idx].reset_index(drop=True)
word_counts = X_binary.getnnz(axis=1)
valid_mask = word_counts >= 5
X_filtered = X_binary[valid_mask]
y_filtered = y_binary[valid_mask]
time_filtered = time_binary[valid_mask].reset_index(drop=True)


#處理時間資料
df_all = pd.DataFrame({
    "id": time_filtered["id"].values,
    "datetime": pd.to_datetime(time_filtered["datetime"].values),
    "label": y_filtered
})
df_all["row_id"] = df_all.index
df_all["datetime"] = pd.Series(time_filtered["datetime"].values).dt.floor("D")
df_all.set_index("datetime", inplace=True)


print(f" 時間範圍：{df_all.index.min()} ~ {df_all.index.max()}")
print(f" 資料筆數：{len(df_all)}")

#使用前三好的模型組合
rf = RandomForestClassifier(n_estimators=200, max_depth=None, class_weight="balanced", random_state=42)
nb = MultinomialNB(alpha=0.5)
xgb = XGBClassifier(n_estimators=200, max_depth=6, use_label_encoder=False, eval_metric='logloss')

voting_model = VotingClassifier(
    estimators=[('rf', rf), ('nb', nb), ('xgb', xgb)],
    voting='soft',
    n_jobs=-1
)

# 移動回測參數（信心0.2)
confidence_threshold = 0.2
start_date = df_all.index.min() + pd.DateOffset(months=3) #時間戳記的加減
end_date = df_all.index.max()
results = []


print(" 開始每日移動回測")
for current_day in tqdm(pd.date_range(start=start_date, end=end_date, freq="D")):
    train_start = current_day - pd.DateOffset(months=3)
    train_end = current_day - pd.DateOffset(days=1)
    test_day = current_day

    train_ids = df_all[(df_all.index >= train_start) & (df_all.index <= train_end)]["row_id"].values
    test_ids = df_all[df_all.index == test_day]["row_id"].values


    if len(train_ids) == 0 or len(test_ids) == 0:
        continue  # 若無資料，跳過

    X_train = X_filtered[train_ids].toarray()
    y_train = y_filtered[train_ids]
    X_test = X_filtered[test_ids].toarray()

    voting_model.fit(X_train, y_train)
    probas = voting_model.predict_proba(X_test)

    for i, idx in enumerate(test_ids):
        prob = probas[i]
        prob_diff = abs(prob[1] - prob[0])
        true_label = y_filtered[idx]
        if prob_diff < confidence_threshold:
            results.append({"id": idx, "predict": "no_action", "true": true_label, "datetime": df_all.iloc[idx].name})
        else:
            pred_label = voting_model.predict(X_filtered[idx].toarray())[0]
            results.append({"id": idx, "predict": pred_label, "true": true_label, "datetime": df_all.iloc[idx].name})

#結果
result_df = pd.DataFrame(results)
out_df = result_df[result_df["predict"] != "no_action"].copy()
out_df["predict"] = out_df["predict"].astype(int)
out_df["true"] = out_df["true"].astype(int)

y_true = out_df["true"]
y_pred = out_df["predict"]


print("\n Classification Report（出手樣本）:")
print(classification_report(y_true, y_pred, target_names=["漲 (1)", "跌 (0)"]))
print(f"\n出手次數：{len(out_df)} / 全部預測次數：{len(result_df)}")
print(f"Macro F1-score: {f1_score(y_true, y_pred, average='macro'):.4f}")
print(f"Accuracy: {accuracy_score(y_true, y_pred):.4f}")

cm = confusion_matrix(y_true, y_pred, labels=[1, 0])
acc = accuracy_score(y_true, y_pred)

table_data = [
        ["", "真實為漲", "真實為跌"],
        ["預測為漲", cm[0][0], cm[0][1]],
        ["預測為跌", cm[1][0], cm[1][1]]]


fig, ax = plt.subplots(figsize=(4, 2.5))
ax.axis("off")
table = ax.table(cellText=table_data, loc="center", cellLoc="center", colWidths=[0.2]*3)
table.auto_set_font_size(False)
table.set_fontsize(12)
table.scale(1.2, 1.2)
plt.title(f"正確率: {acc*100:.2f}%", fontsize=14, color="red", pad=10)
plt.show()


In [ ]:
import matplotlib.pyplot as plt
#結果
result_df = pd.DataFrame(results)
out_df = result_df[result_df["predict"] != "no_action"].copy()
out_df["predict"] = out_df["predict"].astype(int)
out_df["true"] = out_df["true"].astype(int)

y_true = out_df["true"]
y_pred = out_df["predict"]


print("\n Classification Report（出手樣本）:")
print(classification_report(y_true, y_pred, target_names=["漲 (1)", "跌 (0)"]))
print(f"\n出手次數：{len(out_df)} / 全部預測次數：{len(result_df)}")
print(f"Macro F1-score: {f1_score(y_true, y_pred, average='macro'):.4f}")
print(f"Accuracy: {accuracy_score(y_true, y_pred):.4f}")

cm = confusion_matrix(y_true, y_pred, labels=[1, 0])
acc = accuracy_score(y_true, y_pred)

table_data = [
        ["", "真實為漲", "真實為跌"],
        ["預測為漲", cm[0][0], cm[0][1]],
        ["預測為跌", cm[1][0], cm[1][1]]]


fig, ax = plt.subplots(figsize=(4, 2.5))
ax.axis("off")
table = ax.table(cellText=table_data, loc="center", cellLoc="center", colWidths=[0.2]*3)
table.auto_set_font_size(False)
table.set_fontsize(12)
table.scale(1.2, 1.2)
plt.title(f"正確率: {acc*100:.2f}%", fontsize=14, color="red", pad=10)
plt.show()

In [ ]:
daily_acc = out_df.groupby("datetime").apply(lambda g: (g["true"] == g["predict"]).mean())

plt.figure(figsize=(12, 5))
plt.plot(daily_acc.index, daily_acc.values, label="每日準確率", color="royalblue")
plt.axhline(y=daily_acc.mean(), color="brown", linestyle="--", label=f"平均準確率：{daily_acc.mean():.3f}")
plt.title("每日預測準確率（僅含出手樣本）")
plt.ylabel("Accuracy")
plt.xlabel("Date")
plt.xticks(rotation=45)
plt.ylim(0, 1.05)
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
from scipy.sparse import load_npz
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.naive_bayes import MultinomialNB
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score
from tqdm import tqdm

#資料（只保留漲跌）刪除短文章
X_sparse = load_npz("5d_3_trend_vector_model.npz")
y_df = pd.read_csv("5d_3_trend_y.csv", header=None).rename(columns={0: "trend"})
time_df = pd.read_csv("5d_3_trend_time.csv", header=None).rename(columns={0: "datetime"})
time_df.columns = ["datetime"]
time_df["datetime"] = pd.to_datetime(time_df["datetime"])
time_df["id"] = time_df.index

valid_idx = y_df[y_df["trend"].isin([-1, 1])].index
X_binary = X_sparse[valid_idx]
y_binary = y_df.loc[valid_idx, "trend"].replace({-1: 0, 1: 1}).values
time_binary = time_df.loc[valid_idx].reset_index(drop=True)
word_counts = X_binary.getnnz(axis=1)
valid_mask = word_counts >= 5
X_filtered = X_binary[valid_mask]
y_filtered = y_binary[valid_mask]
time_filtered = time_binary[valid_mask].reset_index(drop=True)


#處理時間資料
df_all = pd.DataFrame({
    "id": time_filtered["id"].values,
    "datetime": pd.to_datetime(time_filtered["datetime"].values),
    "label": y_filtered
})
df_all["row_id"] = df_all.index
df_all["datetime"] = pd.Series(time_filtered["datetime"].values).dt.floor("D")
df_all.set_index("datetime", inplace=True)

print(f"時間範圍：{df_all.index.min()} ~ {df_all.index.max()}")
print(f"資料筆數：{len(df_all)}")

#使用前三好的模型
rf = RandomForestClassifier(n_estimators=200, max_depth=None, class_weight="balanced", random_state=42)
nb = MultinomialNB(alpha=0.5)
xgb = XGBClassifier(n_estimators=200, max_depth=6, use_label_encoder=False, eval_metric='logloss')

voting_model = VotingClassifier(
    estimators=[('rf', rf), ('nb', nb), ('xgb', xgb)],
    voting='soft',
    n_jobs=-1
)

#移動回測參數（信心0.3）
confidence_threshold = 0.3
start_date = df_all.index.min() + pd.DateOffset(months=3)
end_date = df_all.index.max()
results = []

print("開始每日移動回測")
for current_day in tqdm(pd.date_range(start=start_date, end=end_date, freq="D")):
    train_start = current_day - pd.DateOffset(months=3)
    train_end = current_day - pd.DateOffset(days=1)
    test_day = current_day

    train_ids = df_all[(df_all.index >= train_start) & (df_all.index <= train_end)]["row_id"].values
    test_ids = df_all[df_all.index == test_day]["row_id"].values


    if len(train_ids) == 0 or len(test_ids) == 0:
        continue  #無資料跳過

    X_train = X_filtered[train_ids].toarray()
    y_train = y_filtered[train_ids]
    X_test = X_filtered[test_ids].toarray()

    voting_model.fit(X_train, y_train)
    probas = voting_model.predict_proba(X_test)

    for i, idx in enumerate(test_ids):
        prob = probas[i]
        prob_diff = abs(prob[1] - prob[0])
        true_label = y_filtered[idx]
        if prob_diff < confidence_threshold:
            results.append({"id": idx, "predict": "no_action", "true": true_label, "datetime": df_all.iloc[idx].name})
        else:
            pred_label = voting_model.predict(X_filtered[idx].toarray())[0]
            results.append({"id": idx, "predict": pred_label, "true": true_label, "datetime": df_all.iloc[idx].name})

#結果統計與顯示
result_df = pd.DataFrame(results)
out_df = result_df[result_df["predict"] != "no_action"].copy()
out_df["predict"] = out_df["predict"].astype(int)
out_df["true"] = out_df["true"].astype(int)

y_true = out_df["true"]
y_pred = out_df["predict"]


print("\nClassification Report（出手樣本）:")
print(classification_report(y_true, y_pred, target_names=["漲 (1)", "跌 (0)"]))
print(f"\n出手次數：{len(out_df)} / 全部預測次數：{len(result_df)}")
print(f"Macro F1-score: {f1_score(y_true, y_pred, average='macro'):.4f}")
print(f"Accuracy: {accuracy_score(y_true, y_pred):.4f}")
cm = confusion_matrix(y_true, y_pred, labels=[1, 0])
acc = accuracy_score(y_true, y_pred)

table_data = [
        ["", "真實為漲", "真實為跌"],
        ["預測為漲", cm[0][0], cm[0][1]],
        ["預測為跌", cm[1][0], cm[1][1]]]


fig, ax = plt.subplots(figsize=(4, 2.5))
ax.axis("off")
table = ax.table(cellText=table_data, loc="center", cellLoc="center", colWidths=[0.2]*3)
table.auto_set_font_size(False)
table.set_fontsize(12)
table.scale(1.2, 1.2)
plt.title(f"正確率: {acc*100:.2f}%", fontsize=14, color="red", pad=10)
plt.show()

In [ ]:
daily_acc = out_df.groupby("datetime").apply(lambda g: (g["true"] == g["predict"]).mean())

plt.figure(figsize=(12, 5))
plt.plot(daily_acc.index, daily_acc.values, label="每日準確率", color="royalblue")
plt.axhline(y=daily_acc.mean(), color="brown", linestyle="--", label=f"平均準確率：{daily_acc.mean():.3f}")
plt.title("每日預測準確率（僅含出手樣本）")
plt.ylabel("Accuracy")
plt.xlabel("Date")
plt.xticks(rotation=45)
plt.ylim(0, 1.05)
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
from scipy.sparse import load_npz
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.naive_bayes import MultinomialNB
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score
from tqdm import tqdm
from scipy.sparse import hstack, csr_matrix
#資料（只保留漲跌）刪除短文章，加入資料來源
X_sparse = load_npz("5d_3_trend_vector_model.npz")
y_df = pd.read_csv("5d_3_trend_y.csv", header=None).rename(columns={0: "trend"})
time_df = pd.read_csv("5d_3_trend_time.csv", header=None).rename(columns={0: "datetime"})
time_df.columns = ["datetime"]
time_df["datetime"] = pd.to_datetime(time_df["datetime"])
time_df["id"] = time_df.index
source_df = pd.read_csv("5d_3_trend_source.csv", header=None).rename(columns={0: "source"})


source_df = pd.get_dummies(source_df["source"])
source_sparse = csr_matrix(source_df.values)
X_combined = hstack([X_sparse, source_sparse])


valid_idx = y_df[y_df["trend"].isin([-1, 1])].index
X_binary = X_combined[valid_idx]
y_binary = y_df.loc[valid_idx, "trend"].replace({-1: 0, 1: 1}).values
time_binary = time_df.loc[valid_idx].reset_index(drop=True)
word_counts = X_binary.getnnz(axis=1)
valid_mask = word_counts >= 5
X_filtered = X_binary[valid_mask]
y_filtered = y_binary[valid_mask]
time_filtered = time_binary[valid_mask].reset_index(drop=True)

#處理時間資料
df_all = pd.DataFrame({
    "id": time_filtered["id"].values,
    "datetime": pd.to_datetime(time_filtered["datetime"].values),
    "label": y_filtered
})
df_all["row_id"] = df_all.index
df_all["datetime"] = pd.Series(time_filtered["datetime"].values).dt.floor("D")
df_all.set_index("datetime", inplace=True)

print(f" 時間範圍：{df_all.index.min()} ~ {df_all.index.max()}")
print(f" 資料筆數：{len(df_all)}")

#使用表現前三好的模型
rf = RandomForestClassifier(n_estimators=200, max_depth=None, class_weight="balanced", random_state=42)
nb = MultinomialNB(alpha=0.5)
xgb = XGBClassifier(n_estimators=200, max_depth=6, use_label_encoder=False, eval_metric='logloss')

voting_model = VotingClassifier(
    estimators=[('rf', rf), ('nb', nb), ('xgb', xgb)],
    voting='soft',
    n_jobs=-1
)

# 移動回測 （信心0.2）
confidence_threshold = 0.2
start_date = df_all.index.min() + pd.DateOffset(months=3)
end_date = df_all.index.max()
results = []


print("開始每日移動回測")
for current_day in tqdm(pd.date_range(start=start_date, end=end_date, freq="D")):
    train_start = current_day - pd.DateOffset(months=3)
    train_end = current_day - pd.DateOffset(days=1)
    test_day = current_day

    train_ids = df_all[(df_all.index >= train_start) & (df_all.index <= train_end)]["row_id"].values
    test_ids = df_all[df_all.index == test_day]["row_id"].values


    if len(train_ids) == 0 or len(test_ids) == 0:
        continue  # 若無資料，跳過

    X_train = X_filtered[train_ids].toarray()
    y_train = y_filtered[train_ids]
    X_test = X_filtered[test_ids].toarray()

    voting_model.fit(X_train, y_train)
    probas = voting_model.predict_proba(X_test)

    for i, idx in enumerate(test_ids):
        prob = probas[i]
        prob_diff = abs(prob[1] - prob[0])
        true_label = y_filtered[idx]
        if prob_diff < confidence_threshold:
            results.append({"id": idx, "predict": "no_action", "true": true_label, "datetime": df_all.iloc[idx].name})
        else:
            pred_label = voting_model.predict(X_filtered[idx].toarray())[0]
            results.append({"id": idx, "predict": pred_label, "true": true_label, "datetime": df_all.iloc[idx].name})

#結果
result_df = pd.DataFrame(results)
out_df = result_df[result_df["predict"] != "no_action"].copy()
out_df["predict"] = out_df["predict"].astype(int)
out_df["true"] = out_df["true"].astype(int)

y_true = out_df["true"]
y_pred = out_df["predict"]


print("\n Classification Report（出手樣本）:")
print(classification_report(y_true, y_pred, target_names=["漲 (1)", "跌 (0)"]))
print(f"\n出手次數：{len(out_df)} / 全部預測次數：{len(result_df)}")
print(f"Macro F1-score: {f1_score(y_true, y_pred, average='macro'):.4f}")
print(f"Accuracy: {accuracy_score(y_true, y_pred):.4f}")

cm = confusion_matrix(y_true, y_pred, labels=[1, 0])
acc = accuracy_score(y_true, y_pred)

table_data = [
        ["", "真實為漲", "真實為跌"],
        ["預測為漲", cm[0][0], cm[0][1]],
        ["預測為跌", cm[1][0], cm[1][1]]]


fig, ax = plt.subplots(figsize=(4, 2.5))
ax.axis("off")
table = ax.table(cellText=table_data, loc="center", cellLoc="center", colWidths=[0.2]*3)
table.auto_set_font_size(False)
table.set_fontsize(12)
table.scale(1.2, 1.2)
plt.title(f"正確率: {acc*100:.2f}%", fontsize=14, color="red", pad=10)
plt.show()

In [ ]:
daily_acc = out_df.groupby("datetime").apply(lambda g: (g["true"] == g["predict"]).mean())

plt.figure(figsize=(12, 5))
plt.plot(daily_acc.index, daily_acc.values, label="每日準確率", color="royalblue")
plt.axhline(y=daily_acc.mean(), color="brown", linestyle="--", label=f"平均準確率：{daily_acc.mean():.3f}")
plt.title("每日預測準確率（僅含出手樣本）")
plt.ylabel("Accuracy")
plt.xlabel("Date")
plt.xticks(rotation=45)
plt.ylim(0, 1.05)
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend()
plt.tight_layout()
plt.show()